<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 5 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Batch Loading, Failures, and Retries</div>
</div>

Ten simulated new orders arrive as a CSV file. You load them with a Stream Load request that you can read line by line, check its response and the table, resend the batch with the same label, and see a batch with one bad row fail as a whole. Then you send the same orders one at a time, without and with Group Commit, and compare what each way leaves in storage. Finally you load the ten tables of the WWI history from Parquet and check them. Plan about 45 minutes.

**Module 5: Batch and Continuous Data Ingestion** · [Lesson](course.md) · [Quiz](quiz05_load_methods_and_retry_safety.ipynb) · [Course README](../../README.md)

## Lab scope

The Lab rebuilds `orders_imported`, `orders_one_by_one`, `orders_group_commit`, and the ten WWI history tables `wwi_orders`, `wwi_order_lines`, `wwi_customers`, `wwi_products`, `wwi_invoices`, `wwi_invoice_lines`, `wwi_customer_transactions`, `wwi_payment_methods`, `wwi_transaction_types`, and `wwi_delivery_methods`. The practice rebuilds `orders_mapping_practice`. It needs the course sandbox from Lab 1, and not the lake environment of Lab 4.

The simulated CSV and the WWI Parquet package are in the repository; the package is unpacked and checked against the course manifest on first use. The course README's [Dataset](../../README.md#dataset) section describes them.

In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, expect
from dw_course.schema import ORDER_COLUMNS, order_ddl
from dw_course.ui import show_sql, show_response, show_records

lab = connect_sandbox()

## 1. Load ten simulated new orders

Orders 900001–900010 are `COURSE_SIMULATION` orders that total 1400.00. This first load writes out the whole HTTP request, so that you can read each part:

1. The URL names the table, `orders_imported` in your lab database, on the BE's HTTP port. The connection step put that address in `DW_BE_HTTP_URL`.
2. The headers carry the settings: the format, the separator, `columns` in the file's order, a **label** for this batch, `strict_mode`, `max_filter_ratio` = 0, and `group_commit` = `off_mode`.
3. `requests.put` sends the file's bytes as the body, with the connection's user and password.
4. `raise_for_status()` stops on an HTTP error, `response.json()` reads Doris's answer, and the checks compare its counts and then the table.

Stream Load usually goes to the FE, which redirects the client to a BE. In this sandbox, the BE registered its address inside the container, `127.0.0.1:8040`, so a redirect would send your client to a port that is not the BE on your machine. The Lab therefore sends the request straight to the BE's published port, 51040, and refuses to follow redirects. Keep the label: step 2 resends the batch with it.

In [ ]:
import os
import requests

lab.execute("DROP TABLE IF EXISTS orders_imported")
ddl = order_ddl("orders_imported")
show_sql("Table creation SQL", ddl)
lab.execute(ddl)
label = "dw_l1_" + uuid4().hex
columns = ",".join(ORDER_COLUMNS)
url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_imported/_stream_load"
headers = {
    "label": label, "format": "csv", "column_separator": ",",
    "columns": columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with (COURSE_ROOT / "datasets/orders.csv").open("rb") as payload:
    response = requests.put(url, auth=(lab.user, lab.password),
                            headers=headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result)
expect((result["Status"], result["NumberLoadedRows"], result["NumberFilteredRows"]), ("Success", 10, 0),
       title="The load reports ten rows loaded and none filtered")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")],
       title="The table holds ten orders, 1400.00")
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_imported ORDER BY order_id", title="Ten loaded new orders");

**Expected result**

The table definition, then the response as a table of fields: `Status` Success, `NumberTotalRows` 10, `NumberLoadedRows` 10, `NumberFilteredRows` 0, the `Label` that you sent, and a `TxnId`, the load's transaction. Two green checks, and the ten orders 900001–900010: order 900001 belongs to customer 1 and amounts to 100.00.

## 2. Resend the same batch with the same label

Pretend that the answer to step 1 was lost, and resend the same file with the same label. `lab.stream_load()` sends the same request as step 1, with the same address, credentials, columns, and settings; from here on the Lab uses it.

Doris remembers the labels of finished loads, those of Stream Loads for 12 hours, so a retry of a batch that committed loads nothing. A new label would be a new batch.

In [ ]:
retry = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/orders.csv", label, columns)
show_response(retry)
expect(retry["Status"], "Label Already Exists", title="Doris recognizes the batch")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")],
       title="The table still holds ten orders, 1400.00")

**Expected result**

`Status` Label Already Exists, with `ExistingJobStatus` FINISHED: the first attempt committed, and this one loaded nothing. The table still holds ten orders and 1400.00.

## 3. Load a batch with one bad row

Load `malformed_orders.csv` with a new label: two orders, the second with the amount `not-a-number`. With `strict_mode` and `max_filter_ratio` = 0, one rejected row fails the whole batch, so the first, valid row is not loaded either.

In [ ]:
rejected = lab.stream_load("orders_imported", COURSE_ROOT / "datasets/malformed_orders.csv",
                           "dw_bad_" + uuid4().hex, columns)
show_response(rejected)
expect((rejected["Status"], rejected["NumberFilteredRows"]), ("Fail", 1), title="The batch fails because of one row")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_imported"), [(10,"1400.00")],
       title="The table is unchanged")
# ErrorURL names the BE's address inside the container; read the same path from the host's BE port.
error_url = rejected["ErrorURL"]
print(requests.get(os.environ["DW_BE_HTTP_URL"] + error_url[error_url.index("/api/"):], timeout=10).text)
lab.sql("SELECT COUNT(*) AS orders, SUM(order_amount) AS amount FROM orders_imported", title="Original data remains intact after rejection of the invalid batch");

**Expected result**

`Status` Fail, `Message` `[DATA_QUALITY_ERROR]too many filtered rows`, `NumberTotalRows` 2, `NumberLoadedRows` 0, `NumberFilteredRows` 1, and an `ErrorURL`. That URL names the BE's address inside the container, `127.0.0.1:8040`; the cell reads the same path from the host's BE port and prints the reason: `Reason: column(order_amount) value is incorrect while strict mode is true, src value is not-a-number`, with the rejected line. Doris keeps such error logs only for a while: they are clues for troubleshooting, not a record. The table still holds ten orders and 1400.00.

## 4. Many small loads, without and with Group Commit

An application often sends one order at a time. Send the ten orders as ten single-row Stream Loads, twice: into `orders_one_by_one` with `group_commit` = `off_mode`, each with its own label, and into `orders_group_commit` with `async_mode`, without a label, because a Group Commit load cannot carry one. Both tables get `group_commit_interval_ms` = 2000, so that a group commits after 2 seconds instead of the default 10.

For each table, the cell counts the transaction IDs of the ten responses, how many rows a query finds right after the tenth response, how long until all ten are visible, and the tablet's version.

In [ ]:
import time

def load_one_line(table, line, mode):
    """Send one CSV line as its own Stream Load; a Group Commit load may not carry a label."""
    headers = {"format": "csv", "column_separator": ",", "columns": columns,
               "strict_mode": "true", "max_filter_ratio": "0", "group_commit": mode}
    if mode == "off_mode":
        headers["label"] = f"{table}_{uuid4().hex}"
    response = requests.put(f"{os.environ['DW_BE_HTTP_URL']}/api/{lab.database}/{table}/_stream_load",
                            auth=(lab.user, lab.password), headers=headers, data=(line + "\n").encode(),
                            timeout=120, allow_redirects=False)
    response.raise_for_status()
    return response.json()

lines = (COURSE_ROOT / "datasets/orders.csv").read_text().splitlines()
comparison = []
for table, mode in (("orders_one_by_one", "off_mode"), ("orders_group_commit", "async_mode")):
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    lab.execute(order_ddl(table))
    interval = f'ALTER TABLE {table} SET ("group_commit_interval_ms" = "2000")'
    show_sql(f"{table}: commit a group after 2 seconds", interval)
    lab.execute(interval)
    responses = [load_one_line(table, line, mode) for line in lines]
    sent = time.monotonic()
    expect([response["Status"] for response in responses], ["Success"] * 10, title=f"{mode}: ten loads succeed")
    visible_at_once = lab.query(f"SELECT COUNT(*) FROM {table}")[0][0]
    while lab.query(f"SELECT COUNT(*) FROM {table}")[0][0] < 10:
        if time.monotonic() - sent > 60:
            raise TimeoutError(f"The rows of {table} were not visible after 60 seconds")
        time.sleep(0.2)
    comparison.append({
        "table": table, "group_commit": mode, "loads": len(responses),
        "transactions": len({response["TxnId"] for response in responses}),
        "visible right after the last answer": visible_at_once,
        "seconds until all 10 visible": round(time.monotonic() - sent, 1),
        "tablet version": lab.query(f"SHOW TABLETS FROM {table}")[0][4],
    })
show_response(responses[-1], title="The last async_mode response")
show_records("Ten single-row loads, without and with Group Commit", comparison)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM orders_group_commit"), [(10, "1400.00")],
       title="Group Commit loaded the same ten orders, 1400.00")

**Expected result**

Two `ALTER TABLE` statements, two green checks that all ten loads of each kind succeeded, the last `async_mode` response, a green check that Group Commit loaded the same ten orders, 1400.00, and the comparison:

| table | group_commit | transactions | visible right after the last answer | seconds until all 10 visible | tablet version |
| --- | --- | ---: | ---: | ---: | ---: |
| orders_one_by_one | off_mode | 10 | 10 | 0.0 | 11 |
| orders_group_commit | async_mode | 1 | 0 | about 2 | 2 |

Without Group Commit, ten loads are ten transactions, each visible when it answers, and ten new versions that compaction will have to merge. With `async_mode`, the ten answers came as soon as each row was in the write-ahead log, all carry the same transaction ID, and the rows appeared together when the group committed about 2 seconds later; the tablet got one version. Rarely, when a group's 2 seconds end while the ten requests are still arriving, there are two transactions and version 3. The `async_mode` response shows `GroupCommit` true and a `Label` that Doris generated, `group_commit_…`. The last check finds the same ten orders and 1400.00.

## 5. Load the ten tables of the WWI history

Use the same kind of request for the WWI history in Parquet: ten business tables, 701,846 rows in all. The cell first checks every file against the course manifest, so that a missing or changed file stops it before any table is dropped. For each table it shows the definition from the manifest and the load's response, and checks the row count and that the primary key is unique. A Parquet file names its fields, so these loads need no `columns`.

In [ ]:
from dw_course.wwi import manifest, parquet_paths, parquet_ddl
paths = parquet_paths()  # Missing files or checksum mismatches stop execution before tables are dropped
for name, metadata in manifest()["tables"].items():
    target = "wwi_" + name
    lab.execute("DROP TABLE IF EXISTS " + target)
    ddl = parquet_ddl(name, target)
    show_sql("WWI historical table: " + name, ddl)
    lab.execute(ddl)
    response = lab.stream_load(target, paths[name], "wwi_" + uuid4().hex, format="parquet")
    show_response(response)
    expect((response["Status"], response["NumberLoadedRows"], response["NumberFilteredRows"]),
           ("Success", metadata["rows"], 0))
    primary = metadata["primary"]
    expect(lab.query(f"SELECT COUNT(*), COUNT(DISTINCT {primary}) FROM {target}"),
           [(metadata["rows"], metadata["rows"])], title=f"{target}: {metadata['rows']:,} rows, one per {primary}")

**Expected result**

Ten definitions and ten responses, each `Success` with no filtered rows, and ten green checks: `wwi_orders` 73,595 rows, `wwi_order_lines` 231,412, `wwi_customers` 663, `wwi_products` 227, `wwi_invoices` 70,510, `wwi_invoice_lines` 228,265, `wwi_customer_transactions` 97,147, `wwi_payment_methods` 4, `wwi_transaction_types` 13, and `wwi_delivery_methods` 10, each with a unique primary key. The first run unpacks the package into `.runtime/wwi/`, which takes a few seconds more.

### Check the order lines and their relationships

A successful load of every table does not prove that the tables fit together. Join the order lines to their orders, customers, and products: if a key were missing or duplicated, the join would lose or multiply lines. Then count orders per day: `COUNT(DISTINCT o.OrderID)` counts orders, because one order has several lines. The [further reading on invoices and account receipts](optional05_invoice_and_receipts.md) compares the other amounts of the history; it is not part of this Lab.

In [ ]:
expect(lab.query("""
SELECT COUNT(*), SUM(l.Quantity*l.UnitPrice)
FROM wwi_order_lines l
JOIN wwi_orders o ON l.OrderID=o.OrderID
JOIN wwi_customers c ON o.CustomerID=c.CustomerID
JOIN wwi_products p ON l.StockItemID=p.StockItemID
"""), [(231412, "177634276.40")], title="Every order line joins one order, one customer, and one product")
lab.sql("""
SELECT o.OrderDate, COUNT(DISTINCT o.OrderID) AS orders,
       SUM(l.Quantity*l.UnitPrice) AS order_amount
FROM wwi_orders o JOIN wwi_order_lines l ON o.OrderID=l.OrderID
GROUP BY o.OrderDate ORDER BY o.OrderDate LIMIT 10
""", title="Daily order analysis across the full history");

**Expected result**

A green check: all 231,412 order lines join one order, one customer, and one product each, and they total 177,634,276.40 before tax. The daily report starts on 2013-01-01 with the first ten days of the history, with dozens to about a hundred orders a day.

## 6. Practice: map columns in another order

[orders_reordered.csv](../../datasets/orders_reordered.csv) contains two new orders. Its first two columns are customer_id and order_id; the remaining columns follow the original CSV order. Independently create orders_mapping_practice and specify the correct columns through HTTP Stream Load. Verify that order 901001 belongs to customer 1 with amount 180, and order 901002 belongs to customer 2 with amount 80.

Follow the HTTP example in step 1, using a new target table, file, and label, and swapping the first two fields in columns. The next cell contains a partly filled template: uncomment it, then fill in the column mapping and the check. It reads the password from the existing connection object rather than writing it in the notebook. This exercise rebuilds only orders_mapping_practice.

The same load can be sent from a terminal in the course directory after the template has created the table. Replace `<columns>` with your mapping; the sandbox accepts Stream Load directly on the BE HTTP port 51040:

```bash
curl -u root: -T datasets/orders_reordered.csv \
  -H "label:mapping_curl_$(date +%s)" -H "format:csv" -H "column_separator:," \
  -H "columns:<columns>" \
  -H "strict_mode:true" -H "max_filter_ratio:0" -H "group_commit:off_mode" \
  http://127.0.0.1:51040/api/dw_course_l1_demo/orders_mapping_practice/_stream_load
```

Write your code in the next cell, then compare it with the reference solution after it.

In [ ]:
# # Select all lines in this cell, uncomment them (Ctrl+/ or Cmd+/), then replace each TODO.
# lab.execute("DROP TABLE IF EXISTS orders_mapping_practice")
# lab.execute(order_ddl("orders_mapping_practice"))
# url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_mapping_practice/_stream_load"
# headers = {
#     "label": "mapping_" + uuid4().hex,
#     "format": "csv",
#     "column_separator": ",",
#     "columns": TODO_COLUMNS,  # TODO: the table column that each file column fills, in file order
#     "strict_mode": "true",
#     "max_filter_ratio": "0",
#     "group_commit": "off_mode",
# }
# with (COURSE_ROOT / "datasets/orders_reordered.csv").open("rb") as payload:
#     response = requests.put(url, auth=(lab.user, lab.password), headers=headers,
#                             data=payload, timeout=120, allow_redirects=False)
# response.raise_for_status()
# show_response(response.json(), title="Column-mapping load result")
# # TODO: query orders_mapping_practice and check the customer and amount of orders 901001 and 901002.


The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
import os
import requests
lab.execute("DROP TABLE IF EXISTS orders_mapping_practice")
ddl = order_ddl("orders_mapping_practice")
show_sql("Exercise table structure", ddl)
lab.execute(ddl)
# The first two input columns are customer ID and order ID; the table uses a different field order.
practice_columns = "customer_id,order_id,order_amount,status,event_version,event_id,event_time,paid_amount,refund_amount,region,data_source"
practice_url = os.environ["DW_BE_HTTP_URL"] + f"/api/{lab.database}/orders_mapping_practice/_stream_load"
practice_headers = {
    "label": "mapping_" + uuid4().hex, "format": "csv", "column_separator": ",",
    "columns": practice_columns, "strict_mode": "true", "max_filter_ratio": "0",
    "group_commit": "off_mode",
}
with (COURSE_ROOT / "datasets/orders_reordered.csv").open("rb") as payload:
    response = requests.put(practice_url, auth=(lab.user, lab.password),
                            headers=practice_headers, data=payload, timeout=120,
                            allow_redirects=False)
response.raise_for_status()
result = response.json()
show_response(result, title="Independent column-mapping load result")
expect(result["Status"], "Success", title="The practice load succeeds")
lab.sql("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id", title="Order ID and customer ID mapping results");
expect(lab.query("SELECT order_id, customer_id, order_amount FROM orders_mapping_practice ORDER BY order_id"),
       [(901001,1,"180.00"),(901002,2,"80.00")], title="Each order has the right customer and amount")

**Expected result**

The table definition, a `Success` response with 2 rows loaded, and the two orders: 901001 for customer 1 with 180.00, and 901002 for customer 2 with 80.00. Without the swapped `columns`, the load would also succeed, with the IDs swapped: order 1 for customer 901001.

## 7. Practice: rank the products by sales

Find the five WWI products with the highest pre-tax sales, `SUM(Quantity * UnitPrice)` over their order lines, with each product's number of lines. Then check that you counted every line exactly once: the sales of all products together must equal the total of all order lines, 177,634,276.40 over 231,412 lines.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
top_products = """SELECT p.StockItemName, COUNT(*) AS order_lines,
       SUM(l.Quantity * l.UnitPrice) AS sales_amount
FROM wwi_order_lines l
JOIN wwi_products p ON l.StockItemID = p.StockItemID
GROUP BY p.StockItemName
ORDER BY sales_amount DESC
LIMIT 5"""
lab.sql(top_products, title="The five products with the highest sales");
expect(lab.query("""SELECT SUM(order_lines), SUM(sales_amount) FROM (
    SELECT COUNT(*) AS order_lines, SUM(l.Quantity * l.UnitPrice) AS sales_amount
    FROM wwi_order_lines l JOIN wwi_products p ON l.StockItemID = p.StockItemID
    GROUP BY p.StockItemID) per_product"""), [(231412, "177634276.40")],
       title="Every order line is counted once")

**Expected result**

Five products, led by `Air cushion machine (Blue)` with 1,061 order lines and 11,107,251.00; the next four, bubble wraps, have about 1,000–1,120 lines and 6.2–6.4 million each. The check is green: the products' sales add up to 177,634,276.40 over 231,412 lines, so every line was counted once. Joining order lines to `wwi_orders` as well would still count each line once, because each line has one order; joining invoice lines too would multiply them.

## You are done when

- `orders_imported` holds ten orders and 1400.00 after one load, a retry with the same label, and a rejected batch;
- you can say why the retry loaded nothing, why the batch with one bad row loaded nothing, and how long a label protects a retry;
- you have compared ten single-row loads without and with Group Commit: their transactions, versions, and when their rows became visible;
- the ten WWI tables hold 701,846 rows, and their order lines join without losing or doubling any.

When a load fails, keep its response and read `ErrorURL` soon. For `Publish Timeout`, or a lost answer, check the original label's outcome before you retry. Module 6 keeps rejected inputs in a table. The other ways to load, from object storage, Kafka, or a database's changes, are in the lesson and the optional reading.

## Next

Open [Quiz 5: Batch and Continuous Data Ingestion](quiz05_load_methods_and_retry_safety.ipynb), then continue to [Module 6: Data Quality and Schema Validation](../module06-data-quality/course.md).